# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset (human mast cell extracellular vesicle mass spectrometry) using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and is accessible from the open data repository as referenced below.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load Croissant metadata and open the dataset using `mlcroissant`. Examine the high-level dataset description and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema location
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Print metadata summary (access as attributes, not dictionary keys)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.date_published}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Let's enumerate the available record sets and their fields. All references below use the canonical `@id`.

We will:
- List all record sets by `@id` and name.
- For each record set, show the available fields and, if present, their `@id` and label.

In [ ]:
# Show dataset record sets and their fields by @id.
# Each record set is identified by its '@id'.
record_sets = dataset.metadata.record_sets
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet: {rs['@id']} | name: {rs.get('name', '')}")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            field_id = field.get('@id', '')
            field_label = field.get('name', field.get('label', ''))
            print(f"    - {field_id} | label: {field_label}")
    print()

## 3. Data Extraction

Let's load data records from the main record set(s). **All references by `@id`.**

- We will extract and inspect the first record set as the main data table in this dataset.

In [ ]:
# Build list of record set @id's for loading, extract the first one for demo.
rs_ids = [rs['@id'] for rs in record_sets]
print(f"RecordSet IDs: {rs_ids}")

dataframes = {}

for record_set_id in rs_ids:
    print(f"Loading records from record set: {record_set_id}")
    # Each element from .records(...) is a dict with field '@id' keys
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame with columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print("No data rows found.")
    print()
# We'll use the first record set with data for further EDA
if dataframes:
    main_record_set_id = next(iter(dataframes))
    main_df = dataframes[main_record_set_id]
    print(f"Selected record set for EDA: {main_record_set_id}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

Process and explore the key numeric data fields.

Examples:
- Filter protein records based on peptide count or abundance.
- Normalize a numeric abundance field.
- Group by a Protein or Sample identifier.

*All field references will use their `@id` values determined above.*

In [ ]:
import numpy as np

if main_record_set_id is not None:
    df = main_df.copy()

    # Identify typical numeric fields (here guessed, change as needed)
    numeric_field_ids = [col for col in df.columns if (df[col].dtype == float or df[col].dtype == int or np.issubdtype(df[col].dtype, np.number))]
    print(f"Numeric field candidates: {numeric_field_ids}")
    if numeric_field_ids:
        numeric_field = numeric_field_ids[0]  # e.g., the first numeric field
        print(f"Example numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()  # just for example
        # Filter records
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records where {numeric_field} > mean ({threshold}):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} (mean=0, std=1):")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by first non-numeric field, if any
        group_candidates = [col for col in df.columns if col != numeric_field and not np.issubdtype(df[col].dtype, np.number)]
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped (mean) {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No non-numeric fields available for grouping.")
    else:
        print("No numeric fields found in the main record set.")
else:
    print("No data extracted to perform EDA.")

## 5. Visualization

Visualize a numeric field distribution (e.g., using a histogram), or plot a relationship between two fields, using field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_ids:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(f"{numeric_field}")
    plt.ylabel("Count")
    plt.show()

    # If more than one numeric field, scatter plot
    if len(numeric_field_ids) > 1:
        plt.figure(figsize=(6,6))
        sns.scatterplot(
            data=main_df,
            x=numeric_field_ids[0],
            y=numeric_field_ids[1]
        )
        plt.title(f"{numeric_field_ids[0]} vs {numeric_field_ids[1]}")
        plt.show()
else:
    print("Visualization skipped - no numeric field data available.")

## 6. Conclusion

- We loaded, inspected, and processed mass spectrometry records from a Croissant FAIR dataset via `mlcroissant`.
- Field and record set references used only `@id` for transparency and reproducibility.
- This example demonstrates a foundation for further proteomics and bioinformatics analysis in Python, leveraging open FAIR standards.

### Next steps
- Explore additional record sets or fields for more detailed statistical or machine learning analysis.
- Join with external protein annotation databases as referenced in the dataset.